# P3-02. 대장내시경 용종 분할 ② U-Net 학습 + 평가

## U-Net이 뭔가요?
의료영상 분할의 표준 모델이에요. 이름처럼 **U자 모양**이에요.

```
입력 256×256 ──────────────────────────────────────────→ 출력 256×256 (픽셀마다 용종 확률)
   │ 인코더(내려가기)                         디코더(올라가기) ↑
   ├─ 128×128 ─────────── 지름길 연결 ──────────────→ 128×128 ┤
   ├── 64×64 ─────────── 지름길 연결 ─────────────→ 64×64 ──┤
   ├─── 32×32 ────────── 지름길 연결 ────────────→ 32×32 ───┤
   └──── 16×16 → 8×8 (가장 압축: "이게 용종이다" 파악) → 키우기 ┘
```
- **인코더**: 프로젝트 1의 ResNet과 똑같이 크기를 줄이며 *"무엇인지"* 파악 → 우리는 **ImageNet 사전학습 ResNet34**를 인코더로 써요
- **디코더**: 다시 키우면서 *"정확히 어디인지"* 복원
- **지름길 연결(skip connection)**: 풀링으로 줄이기 **전의 고해상도 지도를 디코더에 직접 전달** → 프로젝트 1에서 *"풀링하면 정확한 위치를 잃는데, 분할은 어떻게 하지?"*라고 물어보셨던 그 문제의 해결책이에요

## 손실 함수와 평가 지표
| | 내용 | 왜 |
|---|---|---|
| **BCE** | 픽셀마다 "용종일 확률"의 log loss (이진 분류를 픽셀 6만5천 개에 하는 것) | 픽셀 단위로 안정적으로 학습 |
| **Dice loss** | 1 − Dice | 용종이 사진의 10%뿐이라 BCE만 쓰면 "전부 배경"이 쉬운 답이 됨 → **겹치는 정도를 직접 높이게** |
| **Dice** (평가) | 2 × 겹친 넓이 ÷ (정답 넓이 + 예측 넓이) | 0~1, 1이면 완벽. 의료 분할 논문의 표준 지표 |
| **IoU** (평가) | 겹친 넓이 ÷ 합친 넓이 | 프로젝트 2의 IoU와 같은 것 (Dice보다 조금 낮게 나옴) |

**이 노트북에서 답할 질문**
1. U-Net은 용종을 얼마나 정확하게 칠하나? (test Dice, IoU)
2. **사전학습 인코더 vs 처음부터 학습**: 1,000장뿐인 데이터에서 차이가 얼마나 날까? (프로젝트 1 E2 vs E3와 비교)
3. **작은 용종**은 큰 용종보다 얼마나 어려울까?
4. 가장 잘한 사진과 가장 못한 사진은 어떻게 생겼나?

## 💻 컴퓨터 부하 줄이는 설정
- 데이터는 P3-01에서 저장한 **256×256 캐시(.npz)를 메모리에 올려서** 써요 → 학습 중 디스크를 거의 안 읽음
- `num_workers=0` (CPU 병렬 처리 안 함), 배치 8, 혼합 정밀도(AMP)로 GPU 메모리 2~3GB 정도
- **GPU가 2개면 자동으로 두 번째 GPU(`cuda:1`)를 써요** → 첫 번째 GPU의 YOLO 학습과 겹치지 않아요
- GPU가 1개뿐이면 YOLO 학습이 끝난 뒤에 돌리는 걸 추천해요 (같이 돌려도 되지만 둘 다 느려져요)

> 실행 후 **4(학습 곡선), 5, 6, 7번 출력**을 Claude에게 보내주세요.

## 0. 준비 + GPU 선택

In [ ]:
import random, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2
import segmentation_models_pytorch as smp

_avail = {f.name for f in font_manager.fontManager.ttflist}
for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
    if _f in _avail:
        plt.rcParams["font.family"] = _f; break
plt.rcParams["axes.unicode_minus"] = False

SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all()

n_gpu = torch.cuda.device_count()
DEVICE = torch.device("cuda:1" if n_gpu >= 2 else "cuda:0" if n_gpu == 1 else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"GPU {n_gpu}개 → 사용: {DEVICE}" + ("  (YOLO가 cuda:0을 쓰고 있다면 겹치지 않아요)" if n_gpu >= 2 else ""))
if n_gpu == 1:
    print("⚠️ GPU가 1개예요. YOLO 학습 중이면 둘 다 느려질 수 있어요 (같이 돌려도 에러는 안 나요).")

OUT = Path("../../outputs/p3").resolve()
d = np.load(OUT / "kvasir_256.npz", allow_pickle=True)
X, Y, SPLIT = d["X"], d["Y"], d["split"]
AREA = d["area_ratio"]; FILES = d["files"]
for s in ["train", "val", "test"]:
    print(f"{s:<5}: {(SPLIT == s).sum()}장")

## 1. Dataset + 증강

> 🎯 **분할의 증강은 사진과 마스크를 *똑같이* 변형해야 해요.** 사진만 뒤집고 마스크는 그대로면 정답이 엉뚱한 곳을 가리키게 돼요.  
> torchvision의 `tv_tensors.Image`와 `tv_tensors.Mask`로 감싸면 **같은 변형을 한 번에** 적용해줘요. 단, 밝기 변화는 사진에만 적용돼요 (마스크 값은 0/1 그대로).
>
> | 증강 | 이유 |
> |---|---|
> | 좌우·**상하** 반전 | 내시경은 카메라가 돌아가면서 찍어서 **위아래 방향이 정해져 있지 않아요** → 프로젝트 1·2와 달리 상하 반전도 OK |
> | 회전 ±20° | 같은 이유 |
> | 85~100% 확대 자르기 | 거리 차이 흉내 |
> | 밝기·대비 ±20% | 조명·장비 차이 흉내 |

In [ ]:
train_aug = v2.Compose([
    v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(), v2.RandomRotation(20),
    v2.RandomResizedCrop(256, scale=(0.85, 1.0), antialias=True),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
])
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class PolypDS(Dataset):
    def __init__(self, split, aug=None):
        idx = np.where(SPLIT == split)[0]
        self.x = torch.from_numpy(X[idx]).permute(0, 3, 1, 2)          # N×256×256×3 → N×3×256×256
        self.y = torch.from_numpy(Y[idx]).unsqueeze(1)                 # N×1×256×256
        self.aug, self.idx = aug, idx
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        img, msk = tv_tensors.Image(self.x[i]), tv_tensors.Mask(self.y[i])
        if self.aug:
            img, msk = self.aug(img, msk)                              # 사진·마스크에 같은 변형
        img = (img.float() / 255 - MEAN) / STD
        return img, msk.float()

loaders = {s: DataLoader(PolypDS(s, train_aug if s == "train" else None), batch_size=8 if s == "train" else 16,
                         shuffle=(s == "train"), num_workers=0) for s in ["train", "val", "test"]}

ds_demo = PolypDS("train", train_aug)
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for k in range(6):
    im, m = ds_demo[0]
    axes[0, k].imshow(((im * STD + MEAN).permute(1, 2, 0)).clamp(0, 1)); axes[1, k].imshow(m[0], cmap="gray")
for a in axes.ravel(): a.axis("off")
plt.suptitle("같은 사진 1장에 증강 6번: 위 사진과 아래 마스크가 항상 같이 움직여야 해요"); plt.tight_layout(); plt.show()

## 2. 모델, 손실, 지표

> 🎯 `segmentation_models_pytorch`(smp): 분할 모델을 한 줄로 만들어주는 라이브러리예요 (프로젝트 1의 `timm`과 같은 역할).  
> `smp.Unet("resnet34", encoder_weights="imagenet")` → **인코더는 ImageNet 사전학습 ResNet34**, 디코더는 새로 만듦.  
> 출력은 **1채널 256×256**: 픽셀마다 "용종일 점수(logit)" → sigmoid를 거치면 0~1 확률, 0.5 이상을 용종으로 판정해요.

In [ ]:
def make_model(pretrained=True):
    return smp.Unet("resnet34", encoder_weights="imagenet" if pretrained else None, in_channels=3, classes=1)

m = make_model(pretrained=False)
print(f"U-Net(ResNet34) 파라미터: {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M")
with torch.no_grad():
    print("입력 (2, 3, 256, 256) → 출력", tuple(m(torch.randn(2, 3, 256, 256)).shape), "← 픽셀마다 점수 1개")
del m

def dice_loss(logits, y, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3)); union = p.sum((1, 2, 3)) + y.sum((1, 2, 3))
    return (1 - (2 * inter + eps) / (union + eps)).mean()

def loss_fn(logits, y):
    return F.binary_cross_entropy_with_logits(logits, y) + dice_loss(logits, y)

def dice_iou_per_image(pred, y):
    """pred, y: N×1×H×W (0/1) → 사진별 Dice, IoU"""
    inter = (pred * y).sum((1, 2, 3)); ps, ys = pred.sum((1, 2, 3)), y.sum((1, 2, 3))
    dice = torch.where(ps + ys > 0, 2 * inter / (ps + ys), torch.ones_like(inter))
    iou = torch.where(ps + ys - inter > 0, inter / (ps + ys - inter), torch.ones_like(inter))
    return dice, iou

## 3. 학습 (사전학습 vs 처음부터)

> 🎯 **같은 U-Net에서 인코더만 다르게** 두 번 학습해요 (한 번에 한 가지만 바꾸는 비교 — 프로젝트 1 노트북 4의 E2 vs E3와 같아요).  
> 프로젝트 1에선 사전학습이 *성능(+1.6%p)보다 학습 안정성*에서 효과가 컸어요. **데이터가 1,000장뿐인** 이번엔 어떨까요?  
> best 모델은 **val Dice** 기준으로 저장, 10 epoch 개선 없으면 조기 종료. 설정 하나당 수 분~10분 정도 걸려요.

In [ ]:
EPOCHS, PATIENCE = 60, 10
CONFIGS = {"pretrained": True, "scratch": False}

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); dices, ious, probs = [], [], []
    for x, y in loader:
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(x.to(DEVICE))
        p = torch.sigmoid(out.float()).cpu(); probs.append(p)
        dc, io = dice_iou_per_image((p > 0.5).float(), y); dices.append(dc); ious.append(io)
    return torch.cat(dices).numpy(), torch.cat(ious).numpy(), torch.cat(probs)

def train(tag, pretrained):
    seed_all()
    model = make_model(pretrained).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    ckpt = OUT / f"p3_unet_{tag}.pt"; hist, best, wait, t0 = [], -1, 0, time.time()
    for ep in range(1, EPOCHS + 1):
        model.train(); tl = 0
        for x, y in loaders["train"]:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad()
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss = loss_fn(model(x).float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tl += loss.item() * len(x)
        sch.step()
        vd, vi, _ = evaluate(model, loaders["val"])
        hist.append({"epoch": ep, "train_loss": tl / len(loaders["train"].dataset), "val_dice": vd.mean(), "val_iou": vi.mean()})
        if vd.mean() > best:
            best, wait = vd.mean(), 0; torch.save(model.state_dict(), ckpt); flag = " ← best"
        else:
            wait += 1; flag = ""
        print(f"[{tag}][{ep:02d}] train loss {hist[-1]['train_loss']:.3f} | val Dice {vd.mean():.3f} IoU {vi.mean():.3f}{flag}")
        if wait >= PATIENCE:
            print(f"[{tag}] 조기 종료"); break
    print(f"[{tag}] {(time.time() - t0) / 60:.1f}분, best val Dice {best:.3f}\n")
    del model; torch.cuda.empty_cache() if USE_AMP else None
    return pd.DataFrame(hist)

hists = {tag: train(tag, pre) for tag, pre in CONFIGS.items()}

## 4. 학습 곡선

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for tag, h in hists.items():
    axes[0].plot(h["epoch"], h["train_loss"], label=tag); axes[1].plot(h["epoch"], h["val_dice"], label=tag)
axes[0].set_title("train loss (BCE + Dice)"); axes[1].set_title("val Dice (높을수록 좋음)")
for a in axes: a.legend(); a.grid(alpha=.3); a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 5. Test 평가

> 🎯 사진별 Dice를 구해서 **평균**과 **분포**를 같이 봐요. 평균이 높아도 *일부 사진에서 완전히 실패(Dice ≈ 0)*하는 경우가 있는지가 중요해요 (임상에선 그 실패가 용종을 놓친 거니까요).

In [ ]:
results, test_probs = {}, {}
test_idx = np.where(SPLIT == "test")[0]
for tag, pre in CONFIGS.items():
    model = make_model(False).to(DEVICE); model.load_state_dict(torch.load(OUT / f"p3_unet_{tag}.pt", map_location=DEVICE))
    dc, io, pr = evaluate(model, loaders["test"]); test_probs[tag] = pr
    results[tag] = {"Dice 평균": dc.mean(), "Dice 중앙값": np.median(dc), "IoU 평균": io.mean(),
                    "Dice < 0.5 사진 수": int((dc < 0.5).sum()), "Dice < 0.1 (거의 실패)": int((dc < 0.1).sum()), "test 장수": len(dc)}
    results[tag + "_dice"] = dc
    del model
display(pd.DataFrame({k: v for k, v in results.items() if not k.endswith("_dice")}).T.round(3))

plt.figure(figsize=(8, 3.5))
for tag in CONFIGS:
    plt.hist(results[tag + "_dice"], bins=20, range=(0, 1), alpha=.6, label=tag)
plt.xlabel("사진별 test Dice"); plt.ylabel("사진 수"); plt.legend(); plt.title("사진별 Dice 분포"); plt.tight_layout(); plt.show()

## 6. 용종 크기별 성능

> 🎯 **작은 용종은 놓치기 쉽고, 놓치면 임상적으로 문제**예요 (작을 때 떼어내는 게 대장내시경의 목적). 프로젝트 2에서 박스 크기를 봤듯이, 크기 구간별로 Dice를 나눠봐요.  
> 작은 용종에서 Dice가 크게 떨어진다면 → 해상도를 256보다 올리는 실험(384, 512)이 다음 개선 방향이 돼요.

In [ ]:
best_tag = max(CONFIGS, key=lambda t: results[t]["Dice 평균"])
size = pd.cut(AREA[test_idx] * 100, [0, 2, 5, 10, 20, 100], labels=["<2%", "2~5%", "5~10%", "10~20%", "≥20%"])
tbl = pd.DataFrame({"용종 면적": size, **{t: results[t + "_dice"] for t in CONFIGS}}).groupby("용종 면적", observed=True)
display(pd.concat([tbl.size().rename("사진 수"), tbl.mean().round(3)], axis=1))

## 7. 가장 잘한 / 못한 사진 보기

> 🎯 초록 = 전문가 정답 경계, 빨강 = 모델 예측 경계. **못한 사진의 공통점**(작다, 흐리다, 빛 반사, 용종이 여러 개, 납작한 모양 등)을 찾아보세요.

In [ ]:
dc = results[best_tag + "_dice"]; pr = test_probs[best_tag]
order = np.argsort(dc)
picks = list(order[:6]) + list(order[-6:])
fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for ax, k in zip(axes.ravel(), picks):
    i = test_idx[k]
    ax.imshow(X[i]); ax.contour(Y[i], colors="lime", linewidths=1.5); ax.contour(pr[k, 0].numpy() > 0.5, colors="red", linewidths=1.2)
    ax.set_title(f"Dice {dc[k]:.2f} / 면적 {AREA[i]:.1%}", fontsize=9, color="red" if dc[k] < 0.5 else "black"); ax.axis("off")
axes[0, 0].set_ylabel("가장 못한 6장"); axes[1, 0].set_ylabel("가장 잘한 6장")
plt.suptitle(f"{best_tag} 모델 — 위: 가장 못한 6장 / 아래: 가장 잘한 6장 (초록 = 정답, 빨강 = 예측)"); plt.tight_layout(); plt.show()

## ✍️ 정리 & 생각해볼 질문

1. test Dice 평균과 중앙값은? 둘이 다르다면 분포가 어떻게 생겼다는 뜻일까?
2. **사전학습 vs 처음부터**: 차이가 프로젝트 1(E2 0.988 vs E3 0.972)보다 컸나? 데이터 양(1,000장 vs 4,300장)과 관련이 있을까?
3. 용종 크기별로 Dice가 어떻게 달랐나? 가장 작은 구간은 몇 장이었나 (표본이 충분한가)?
4. 가장 못한 사진들의 공통점은?
5. Dice 0.9와 0.8의 차이는 그림으로 보면 어느 정도일까? 임상적으로 의미 있는 차이일까?

**메모:**
- 
